# Home Credit Risk Analysis

## Dataset: application_train.csv

### Objetivo 

Analizar la estructura, calidad y relaciones del dataset principal de solicitudes de crédito para comprender su papel dentro de la arquitectura Lakehouse y del modelo dimensional. 

---

### Contexto del negocio 

Home credit es una entidad financiera que busca predecir el riesgo de incumplimiento de pago de sus clientes. 

Este dataset contiene la información principal de las solicitudes de crédito y será utilizado como punto de partida para la construcción de las capas: 

- Bronze 
- Silver 
- Intermediate 
- Gold 
- Diamond 

Dentro de la arquitectura híbrida ETL/ELT.

# 1. Importación de librerías 

Se cargan las librerías necesarias para el análisis exploratorio de datos. 

In [1]:
import pandas as pd 
import numpy as np 

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# 2. Carga del Dataset 

Se carga el archivo `application_train.csv`, considerado el dataset principal del dominio de negocio. 

Este dataset representa las solicitudes de crédito realizadas por los clientes. 

In [2]:
df = pd.read_csv("../../../data/raw/homeCredit/application_train.csv")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/raw/homeCredit/application_train.csv'

### Observaciones iniciales 

El dataset contiene **307.511 solicitudes de crédito** y **122 variables**. 

La cantidad de atributos sugiere que la información proviene de múltiples dominios de negocio: 

- Información demográfica 
- Información laboral 
- Información financiera 
- Información patrimonial 
- Información crediticia 
- Información histórica de comportamiento 

Debido a su natiraleza central, se espera que este dataset actúe cómo la principal tabla de hechos del modelo dimensiona: 

**FACT_CREDIT_APPLICATION**

# 3. Vista preliminar de los datos 

Se inspeccionan los primeros registros para comprender la estructura general y el tipo de información almacenada.

In [3]:
df.head()

NameError: name 'df' is not defined

# 4. Estructura General 

Se analiza la composición del dataset: 

- Cantidad de columnas 
- Tipos de datos 
- Presencia de valores nulos 
- Distribución general de atributos 

In [4]:
df.info()

NameError: name 'df' is not defined

# 5. Identificación de la Clave Primaria 

Se evalúa si la columna `SK_ID_CURR` puede actuar como identificador único de cada solicitud. 

Esta validación es fundamental para: 

- Integridad de datos 
- Relaciones entre datasets 
- Diseño dimensional 
- Procesos de deduplicación 

In [5]:
total_rows = len(df)

unique_ids = df["SK_ID_CURR"].nunique()

duplicates = df["SK_ID_CURR"].duplicated().sum()

print("Resgistros: ", total_rows)
print("IDs únicos: ", unique_ids)
print("Duplicados: ", duplicates)

NameError: name 'df' is not defined

## Resultados 

| Métricas | Valor |
|----------|-------|
|Total registros | 307,511 |
|IDs únicos | 307,511 |
|IDs duplicados | 0 |

### Conclusión 

La columna SK_ID_CURR identifica de manera única cada registro. 

Por lo tanto: 

- SK_ID_CURR será considerada la clave primaria. 
- No se detectaron duplicados. 
- Cada fila representa una única solicitud de crédito. 

### Implicaciones 

FACT_CREDIT_APPLICACTION

# 6. Análisis de Calidad de Datos 

Se identifican columnas con información faltante. 

Este análisis permitirá posteriormente definir: 

- Reglas de calidad 
- Estrategias de imputación 
- Transformaciones Silver 

In [6]:
null_percentage = (
    df.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

null_percentage.head(20)

NameError: name 'df' is not defined

## Principales columnas afectadas 

Las variables con mayor porcentaje de valores faltantes pertenecen principalmente a información inmobiliaria. 

### Conclusión 

Estas columnas deberán ser evaluadas durante la construcción de la capa Silver mediante reglas de calidad e imputación. 

# 7. Detección de Registros Duplicados 

Se verifica la existencia de registros completamente duplicados dentro del dataset. 

In [7]:
df.duplicated().sum()

NameError: name 'df' is not defined

# 8. Varibale Objetivo 

La variable TARGET representa el comportamiento final del crédito. 

Valores: 

- 0 -> CLiente sin incumplimineto 
- 1 -> Cliente con incumplimiento 

Este atributo será utilizado posteriormente para análisis predictivos. 

In [8]:
df["TARGET"].value_counts()

NameError: name 'df' is not defined

In [9]:
(
    df["TARGET"]
    .value_counts(normalize=True)
    .mul(100)
)

NameError: name 'df' is not defined

TARGET reoresenta el resultado final del crédito. 

| Valor | Significado |
|-------|-------------|
|0| Cliente sin incumplimiento|
|1| Cliente con incumplimineto| 

### Observación 

Existe un desbalance importante entre clases, siendo mucho más frecuente el no incumplimiento. 

# 9. Variables Categóricas 

Se identifican las columnas categóricas presentes en el dataset. 

Estas variables son candidatas a: 

- Dimensiones 
- Catálogos de referencia 
- Transformaciones de codificación 

In [10]:
categorical_columns = df.select_dtypes(include="object").columns

categorical_columns.tolist()

NameError: name 'df' is not defined

# 10. Conclusiones Técnicas 


## Resumen del Dataset 

| Métrica    | Valor      |
|------------|------------|
| Filas      |   307.511  |
| Columnas   |     122    |
| PK         | SK_ID_CURR |
| Duplicados |      0     |
| Target     |   TARGET   |

---

#### Interpretación 

El dataset presenta una alta dimensionalidad, con 122 atributos por registro. 

Cada fila representa una solicitud de crédito realizada por un cliente dentro del programa Home Credit. 

Este dataset constituye la fuente principal de información para el análisis de riesgo crediticio y será utilizado como entidad central dentro del modelo dimensional. 

---

## Calidad de datos 

Se identifican múltiples columnas con porcentajes elevados de valores faltantes, principalmente relacionadas con características inmobiliarias y patrimoniales de los clientes. 

Las columnas más afectadas presentan porcentajes de nulidad superiores al 60%. 

Estas observaciones serán consideradas durante el diseño de las reglas de calidad de la capa Silver. 

---

## Granularidad 

Cada registro representa una única solicitud de crédito asociada a un cliente. 

La columna `SK-ID_CURR` identifica de manera única cada observación. 

--- 

## Relaciones identificadas 

Este dataset actúa como entidad raíz dentro del ecosistema Home Credit y se relaciona con: 

- bureau
- bureau_balance
- previus_application
- installments_payments
- POS_CASH_balance
- credit_card_balance

--- 

## Rol dentro del modelo dimensional 

Entidad principal: 

FACT_CREDIT_APPLICACTION

Desde esta tabla se derivarán las relaciones hacía las dimensiones de cliente, crédito, comportamiento financiero y pagos. 

--- 

## Flujo Lakehouse 

applicaction_train.csv 

↓ Bronze (ingesta cruda)

↓ Silver (calidad y normalización)

↓ Intermediate (integración y enriquecimiento)

↓ Gold (analítica de negocio)

↓ Diamond (convergencia corporativa)